In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.interpolate import splprep, splev

def generate_zamboni_path():

    def get_lap_points(offset=0.0):
        # Top Left
        top_left_x = [
            -28.835 + offset,
            -28.835 + offset,
            -28.65 + (offset * 0.95),
            -27.9 + (offset * 0.80),
            -25.9 + (offset * 0.60),
            -23.0 + (offset * 0.15),
            -20.0,
            -10.0
        ]
        top_left_y = [
            0.0,
            6.0,
            9.0 - (offset * 0.15),
            10.9 - (offset * 0.60),
            12.9 - (offset * 0.80),
            13.735 - (offset * 0.95),
            13.835 - offset,
            13.835 - offset
        ]

        # Top Right
        top_right_x = [
            0.0,
            10.0,
            20.0,
            22.9 - (offset * 0.15),
            25.85 - (offset * 0.60),
            27.9 - (offset * 0.80),
            28.55 - (offset * 0.95),
            28.835 - offset
        ]
        top_right_y = [
            13.835 - offset,
            13.835 - offset,
            13.835 - offset,
            13.735 - (offset * 0.95),
            12.85 - (offset * 0.80),
            10.9 - (offset * 0.60),
            9.0 - (offset * 0.15),
            6.0
        ]

        # Bottom Right
        bottom_right_x = [
            28.835 - offset,
            28.835 - offset,
            28.585 - (offset * 0.95),
            27.85 - (offset * 0.80),
            25.75 - (offset * 0.60),
            22.75 - (offset * 0.15),
            20.0,
            10.0
        ]
        bottom_right_y = [
            0.0,
            -5.75,
            -8.75 + (offset * 0.15),
            -10.75 + (offset * 0.60),
            -13.0 + (offset * 0.80),
            -13.685 + (offset * 0.95),
            -13.835 + offset,
            -13.835 + offset
        ]

        # Bottom Left
        bottom_left_x = [
            0.0,
            -10.0,
            -20.0,
            -22.9 + (offset * 0.15),
            -25.9 + (offset * 0.60),
            -27.9 + (offset * 0.80),
            -28.65 + (offset * 0.95),
            -28.835 + offset
        ]
        bottom_left_y = [
            -13.835 + offset,
            -13.835 + offset,
            -13.835 + offset,
            -13.65 + (offset * 0.95),
            -12.9 + (offset * 0.80),
            -10.9 + (offset * 0.60),
            -8.9 + (offset * 0.15),
            -6.0
        ]

        cx = top_left_x + top_right_x + bottom_right_x + bottom_left_x
        cy = top_left_y + top_right_y + bottom_right_y + bottom_left_y

        return cx, cy

    def get_shifting_sweep_points(offset=0.0, is_last_lap=False, is_first_lap=False):
        """
        Generates the inner closed loop.
        Offset shifts the ENTIRE loop UPWARDS.
        If is_last_lap is True, omits the final left turn.
        """

        if is_first_lap:
            # The very first lap needs to start deep in the center
            pass1_x = [-25.0, -20.0, -10.0, 0.0, 10.0, 20.0]
            pass1_y = [0.75 + offset, 1.0 + offset, 1.0 + offset, 1.0 + offset, 1.0 + offset, 1.0 + offset]
        else:
            # All normal laps start further right to avoid crowding the left boards
            pass1_x = [-20.0, -10.0, 0.0, 10.0, 20.0]
            pass1_y = [1.0 + offset, 1.0 + offset, 1.0 + offset, 1.0 + offset, 1.0 + offset]

        # RIGHT TURN
        right_turn_x = [25.5, 26.5, 25.5]
        right_turn_y = [-0.5 + offset, -5.0 + offset, -9.0 + offset]

        if is_last_lap:
            # Stretch the final return trip all the way to X=-25.0 to close the gap
            pass2_x = [20.0, 10.0, 0.0, -10.0, -20.0, -25.0]
            pass2_y = [-10.4 + offset, -10.4 + offset, -10.4 + offset, -10.4 + offset, -10.4 + offset, -10.4 + offset]
        else:
            # Normal return trips stop at -20.0 to prepare for the left turn
            pass2_x = [20.0, 10.0, 0.0, -10.0, -20.0]
            pass2_y = [-10.4 + offset, -10.4 + offset, -10.4 + offset, -10.4 + offset, -10.4 + offset]

        # LEFT TURN
        left_turn_x = [-25.5, -26.5, -25.5]
        left_turn_y = [-9.0 + offset, -3.0 + offset, 2.3 + offset]

        # STITCHING LOGIC
        if is_last_lap:
            # Drop the left turn completely! The path just ends at X=-20.0 or -25.0
            cx = pass1_x + right_turn_x + pass2_x
            cy = pass1_y + right_turn_y + pass2_y
        else:
            # Build the normal continuous loop
            cx = pass1_x + right_turn_x + pass2_x + left_turn_x
            cy = pass1_y + right_turn_y + pass2_y + left_turn_y
            
        return cx, cy

    # 1. Generate the raw control points for the outer laps
    cx1, cy1 = get_lap_points(offset=0.0)
    cx2, cy2 = get_lap_points(offset=2.0)

    # 2. Generate the center sweeps
    cx_sweep1, cy_sweep1 = get_shifting_sweep_points(offset=0.0, is_last_lap=False, is_first_lap=True)
    cx_sweep2, cy_sweep2 = get_shifting_sweep_points(offset=2.0, is_last_lap=False, is_first_lap=False)
    cx_sweep3, cy_sweep3 = get_shifting_sweep_points(offset=4.0, is_last_lap=False, is_first_lap=False)
    cx_sweep4, cy_sweep4 = get_shifting_sweep_points(offset=6.0, is_last_lap=False, is_first_lap=False)
    cx_sweep5, cy_sweep5 = get_shifting_sweep_points(offset=8.0, is_last_lap=False, is_first_lap=False)
    cx_sweep6, cy_sweep6 = get_shifting_sweep_points(offset=10.0, is_last_lap=True, is_first_lap=False)
    
    # 3. STITCH THEM ALL TOGETHER
    cx_total = cx1 + cx2 + cx_sweep1 + cx_sweep2 + cx_sweep3 + cx_sweep4 + cx_sweep5 + cx_sweep6 
    cy_total = cy1 + cy2 + cy_sweep1 + cy_sweep2 + cy_sweep3 + cy_sweep4 + cy_sweep5 + cy_sweep6 

    # 4. Generate the single spline trajectory
    tck, u = splprep([cx_total, cy_total], s=0, k=2)

    u_fine = np.linspace(0, 1, 750)
    smooth_x, smooth_y = splev(u_fine, tck)

    # --- VISUALIZE IT ---
    fig, ax = plt.subplots(figsize=(14, 7))

    # Updated rink anchor to (-30, -15) so it matches the Gazebo frame!
    rink = patches.FancyBboxPatch((-30, -15), 60, 30, boxstyle="round,pad=0,rounding_size=8.5",
                                  linewidth=2, edgecolor='black', facecolor='aliceblue', zorder=0)
    ax.add_patch(rink)

    ax.plot(smooth_x, smooth_y, 'b-', linewidth=1.5, label='Stitched Trajectory')

    # Stamp the 2-meter Zamboni footprints
    for i in range(len(smooth_x)):
        swept_ice = patches.Circle(
            (smooth_x[i], smooth_y[i]),
            radius=1.0,
            color='cyan',
            alpha=0.3,
            edgecolor='none'
        )
        ax.add_patch(swept_ice)

    ax.plot(cx_total, cy_total, 'ro--', markersize=4, label='Raw Control Points')

    ax.set_title("Gazebo Origin (0,0) Zamboni Pattern")
    ax.axis('equal')
    ax.legend()
    plt.show()

if __name__ == '__main__':
    generate_zamboni_path()

: 